In [2]:
#!/usr/bin/env python3
# Modified script with enhanced visualizations for HAL, KDE, and Kaplan-Meier comparisons
# Added support for simultaneous comparison of smoothness 0 and 1 HAL results
# Full data case analysis

# Import needed libraries
import numpy as np
from numpy import trapezoid  # Use trapezoid instead of trapz
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy.stats import truncnorm, ttest_rel, linregress
from sklearn.neighbors import KernelDensity
from lifelines import KaplanMeierFitter
import time
from matplotlib.gridspec import GridSpec

# Set global parameters
n_experiments = 500  # Total number of experiments in HAL results
n_generate = 500  # Number of experiments to generate for each sample size
ridge = 1e-6
sample_sizes = [200, 400, 800, 1600, 3200]  # All sample sizes to analyze
smoothness_levels = [0, 1]  # Both smoothness levels to include in comparison

# Directory structure adjustment
hal_dir = "../Asymptoticity_FD_0301/out"  # HAL results directory
comparison_base_dir = "."  # Current directory
evaluation_points = np.linspace(0.02, 0.98, 20)

# Set style for plots
sns.set_style("whitegrid")
plt.rcParams.update({'font.size': 12})

# Create containers for cross-sample size summary metrics
# Now separate for each smoothness level
sample_size_summary = {
    'sample_size': [],
    # Survival function metrics
    'hal_init_surv_mean_abs_bias_smooth0': [], 
    'hal_init_surv_mean_abs_bias_smooth1': [],
    'hal_final_surv_mean_abs_bias_smooth0': [],
    'hal_final_surv_mean_abs_bias_smooth1': [],
    'kde_surv_mean_abs_bias': [], 
    'km_surv_mean_abs_bias': [],
    
    'hal_init_surv_mean_variance_smooth0': [], 
    'hal_init_surv_mean_variance_smooth1': [],
    'hal_final_surv_mean_variance_smooth0': [],
    'hal_final_surv_mean_variance_smooth1': [],
    'kde_surv_mean_variance': [], 
    'km_surv_mean_variance': [],
    
    'hal_init_surv_mean_mse_smooth0': [], 
    'hal_init_surv_mean_mse_smooth1': [],
    'hal_final_surv_mean_mse_smooth0': [],
    'hal_final_surv_mean_mse_smooth1': [],
    'kde_surv_mean_mse': [], 
    'km_surv_mean_mse': [],
    
    # Density function metrics - removed final HAL metrics
    'hal_init_dens_mean_abs_bias_smooth0': [], 
    'hal_init_dens_mean_abs_bias_smooth1': [],
    'kde_dens_mean_abs_bias': [],
    
    'hal_init_dens_mean_variance_smooth0': [], 
    'hal_init_dens_mean_variance_smooth1': [],
    'kde_dens_mean_variance': [],
    
    'hal_init_dens_mean_mse_smooth0': [], 
    'hal_init_dens_mean_mse_smooth1': [],
    'kde_dens_mean_mse': []
}

# Function to generate truncated normal data
def generate_truncated_normal(n_samples, mean=0.5, std=0.1, lower=0, upper=1, seed=None):
    if seed is not None:
        np.random.seed(seed)
    a, b = (lower - mean) / std, (upper - mean) / std
    T = truncnorm.rvs(a, b, loc=mean, scale=std, size=n_samples)
    return T

# KDE with bandwidth selection (updated to use trapezoid)
def estimate_density_kde(data, evaluation_points, bandwidth=None):
    """
    Estimate density using KDE with automatic bandwidth selection if not provided.
    """
    if bandwidth is None:
        # Scott's rule for bandwidth selection
        bandwidth = 1.06 * np.std(data) * len(data)**(-1/5)
    
    # Fit KDE
    kde = KernelDensity(bandwidth=bandwidth, kernel='gaussian')
    kde.fit(data.reshape(-1, 1))
    
    # Get log density and convert to density
    log_density = kde.score_samples(evaluation_points.reshape(-1, 1))
    density = np.exp(log_density)
    
    # Normalize to ensure it integrates to 1 over [0,1]
    density = density / trapezoid(density, evaluation_points)
    
    return density, kde

# Compute survival function from KDE
def compute_survival_from_kde(kde, evaluation_points):
    """
    Compute survival function from KDE by numerical integration.
    """
    survival = np.zeros_like(evaluation_points)
    grid = np.linspace(0, 1, 1000)  # Fine grid for integration
    
    log_density_grid = kde.score_samples(grid.reshape(-1, 1))
    density_grid = np.exp(log_density_grid)
    density_grid = density_grid / trapezoid(density_grid, grid)  # Normalize
    
    for i, point in enumerate(evaluation_points):
        # S(x) = integral of f(t) dt from x to 1
        mask = grid >= point
        if any(mask):
            survival[i] = trapezoid(density_grid[mask], grid[mask])
        else:
            survival[i] = 0
    
    return survival

# Kaplan-Meier estimator
def estimate_survival_km(data, evaluation_points):
    """
    Estimate survival function using Kaplan-Meier.
    """
    kmf = KaplanMeierFitter()
    kmf.fit(data)
    survival = kmf.survival_function_at_times(evaluation_points).values
    return survival.flatten()

# Function to compute true density and survival
def compute_true_functions(evaluation_points, mean=0.5, std=0.1, lower=0, upper=1):
    """
    Compute true density and survival for truncated normal.
    """
    a, b = (lower - mean) / std, (upper - mean) / std
    true_density = truncnorm.pdf(evaluation_points, a, b, loc=mean, scale=std)
    true_density = true_density / trapezoid(true_density, evaluation_points)
    
    true_survival = 1 - truncnorm.cdf(evaluation_points, a, b, loc=mean, scale=std)
    # Adjust survival for truncation
    true_survival = true_survival / truncnorm.sf(0, a, b, loc=mean, scale=std)
    
    return true_density, true_survival

# Modified load_data function that uses separate directories
def load_data_fixed(hal_dir, comp_dir, n_samples, smoothness_order, ridge):
    """
    Modified load_data function that uses separate directories for HAL and comparison data.
    """
    # Load HAL results
    hal_base = f"{hal_dir}/hal_n{n_samples}_smooth{smoothness_order}_ridge{ridge}"
    hal_data = {
        "initial_density": pd.read_csv(f"{hal_base}_initial_density.csv", index_col=0),
        "final_density": pd.read_csv(f"{hal_base}_final_density.csv", index_col=0),
        "initial_survival": pd.read_csv(f"{hal_base}_initial_survival.csv", index_col=0),
        "final_survival": pd.read_csv(f"{hal_base}_final_survival.csv", index_col=0),
        "initial_density_se": pd.read_csv(f"{hal_base}_initial_density_se.csv", index_col=0),
        "initial_survival_se": pd.read_csv(f"{hal_base}_initial_survival_se.csv", index_col=0),
        "initial_survival_se_eic": pd.read_csv(f"{hal_base}_initial_survival_se_eic.csv", index_col=0),
        "final_survival_se": pd.read_csv(f"{hal_base}_final_survival_se.csv", index_col=0),
        "final_survival_se_eic": pd.read_csv(f"{hal_base}_final_survival_se_eic.csv", index_col=0)
    }
    
    # Load comparison results from separate directory
    comp_base = f"{comp_dir}/comparison_n{n_samples}"
    comparison_data = {
        "kde_density": pd.read_csv(f"{comp_base}_kde_density.csv", index_col=0),
        "kde_survival": pd.read_csv(f"{comp_base}_kde_survival.csv", index_col=0),
        "km_survival": pd.read_csv(f"{comp_base}_km_survival.csv", index_col=0),
        "true_density": pd.read_csv(f"{comp_base}_true_density.csv", index_col=0),
        "true_survival": pd.read_csv(f"{comp_base}_true_survival.csv", index_col=0)
    }
    
    return hal_data, comparison_data

# Function to generate comparison data for a specific sample size
def generate_comparison_data(n_samples, n_generate, evaluation_points, out_comparison_dir):
    """Generate comparison data for a specific sample size"""
    print(f"Generating comparison data for {n_generate} experiments with n={n_samples} samples each...")
    
    # Initialize DataFrames to store results - use n_experiments to match HAL data
    exp_ids = [f"Exp_{i+1}" for i in range(n_experiments)]
    
    # Initialize with NaN to indicate missing data
    kde_density_df = pd.DataFrame(index=evaluation_points, columns=exp_ids)
    kde_survival_df = pd.DataFrame(index=evaluation_points, columns=exp_ids)
    km_survival_df = pd.DataFrame(index=evaluation_points, columns=exp_ids)
    true_density_df = pd.DataFrame(index=evaluation_points, columns=exp_ids)
    true_survival_df = pd.DataFrame(index=evaluation_points, columns=exp_ids)
    
    # Create directory if it doesn't exist
    os.makedirs(out_comparison_dir, exist_ok=True)
    
    # Run experiments sequentially
    for i in range(n_generate):
        if i % 10 == 0:
            print(f"Processing experiment {i+1}/{n_generate}...")
        
        # Set seed
        seed = 12776 + i
        
        # Generate data
        T = generate_truncated_normal(n_samples, seed=seed)
        
        # KDE for density and survival
        kde_density, kde_model = estimate_density_kde(T, evaluation_points)
        kde_survival = compute_survival_from_kde(kde_model, evaluation_points)
        
        # Kaplan-Meier for survival
        km_survival = estimate_survival_km(T, evaluation_points)
        
        # True density and survival for reference
        true_density, true_survival = compute_true_functions(evaluation_points)
        
        # Store results for this experiment
        kde_density_df[f"Exp_{i+1}"] = kde_density
        kde_survival_df[f"Exp_{i+1}"] = kde_survival
        km_survival_df[f"Exp_{i+1}"] = km_survival
        true_density_df[f"Exp_{i+1}"] = true_density
        true_survival_df[f"Exp_{i+1}"] = true_survival
    
    # Save results
    print(f"Saving comparison results to {out_comparison_dir}...")
    
    kde_density_df.to_csv(f"{out_comparison_dir}/comparison_n{n_samples}_kde_density.csv", index=True)
    kde_survival_df.to_csv(f"{out_comparison_dir}/comparison_n{n_samples}_kde_survival.csv", index=True)
    km_survival_df.to_csv(f"{out_comparison_dir}/comparison_n{n_samples}_km_survival.csv", index=True)
    true_density_df.to_csv(f"{out_comparison_dir}/comparison_n{n_samples}_true_density.csv", index=True)
    true_survival_df.to_csv(f"{out_comparison_dir}/comparison_n{n_samples}_true_survival.csv", index=True)
    
    return (kde_density_df, kde_survival_df, km_survival_df, true_density_df, true_survival_df)

# Function to compute performance metrics (bias, variance, MSE) across evaluation points
def compute_performance_metrics(hal_data, comparison_data):
    """
    Compute performance metrics (bias, variance, MSE) for all methods across evaluation points.
    Returns a dictionary with all metrics for both density and survival.
    """
    # Extract evaluation points and common experiment IDs
    eval_points = hal_data["initial_survival"].index.astype(float).values
    hal_exps = set(hal_data["initial_survival"].columns)
    comp_exps = set(comparison_data["km_survival"].columns)
    common_exps = sorted(list(hal_exps.intersection(comp_exps)))
    
    if len(common_exps) == 0:
        print("No common experiments found. Cannot compute metrics.")
        return None
    
    print(f"Computing metrics using {len(common_exps)} common experiments.")
    
    # Get true density and survival functions
    true_density, true_survival = compute_true_functions(eval_points)
    
    # Initialize arrays for metrics
    n_points = len(eval_points)
    
    # ----- SURVIVAL FUNCTION METRICS -----
    # Bias calculation for survival
    hal_init_surv_bias = np.zeros(n_points)
    hal_final_surv_bias = np.zeros(n_points)
    kde_surv_bias = np.zeros(n_points)
    km_surv_bias = np.zeros(n_points)
    
    # Variance calculation for survival
    hal_init_surv_variance = np.zeros(n_points)
    hal_final_surv_variance = np.zeros(n_points)
    kde_surv_variance = np.zeros(n_points)
    km_surv_variance = np.zeros(n_points)
    
    # MSE calculation for survival
    hal_init_surv_mse = np.zeros(n_points)
    hal_final_surv_mse = np.zeros(n_points)
    kde_surv_mse = np.zeros(n_points)
    km_surv_mse = np.zeros(n_points)
    
    # Extract data for common experiments - Survival
    hal_init_surv_data = hal_data["initial_survival"][common_exps].values
    hal_final_surv_data = hal_data["final_survival"][common_exps].values
    kde_surv_data = comparison_data["kde_survival"][common_exps].values
    km_surv_data = comparison_data["km_survival"][common_exps].values
    
    # ----- DENSITY FUNCTION METRICS -----
    # Bias calculation for density
    hal_init_dens_bias = np.zeros(n_points)
    hal_final_dens_bias = np.zeros(n_points)
    kde_dens_bias = np.zeros(n_points)
    
    # Variance calculation for density
    hal_init_dens_variance = np.zeros(n_points)
    hal_final_dens_variance = np.zeros(n_points)
    kde_dens_variance = np.zeros(n_points)
    
    # MSE calculation for density
    hal_init_dens_mse = np.zeros(n_points)
    hal_final_dens_mse = np.zeros(n_points)
    kde_dens_mse = np.zeros(n_points)
    
    # Extract data for common experiments - Density
    hal_init_dens_data = hal_data["initial_density"][common_exps].values
    hal_final_dens_data = hal_data["final_density"][common_exps].values
    kde_dens_data = comparison_data["kde_density"][common_exps].values
    
    # Compute metrics for survival function
    for i in range(n_points):
        # Extract survival estimates at this evaluation point
        hal_init_surv_estimates = hal_init_surv_data[i, :]
        hal_final_surv_estimates = hal_final_surv_data[i, :]
        kde_surv_estimates = kde_surv_data[i, :]
        km_surv_estimates = km_surv_data[i, :]
        
        # True survival value at this point
        true_surv_value = true_survival[i]
        
        # Compute bias (mean estimate - true value) - Survival
        hal_init_surv_bias[i] = np.mean(hal_init_surv_estimates) - true_surv_value
        hal_final_surv_bias[i] = np.mean(hal_final_surv_estimates) - true_surv_value
        kde_surv_bias[i] = np.mean(kde_surv_estimates) - true_surv_value
        km_surv_bias[i] = np.mean(km_surv_estimates) - true_surv_value
        
        # Compute variance - Survival
        hal_init_surv_variance[i] = np.var(hal_init_surv_estimates)
        hal_final_surv_variance[i] = np.var(hal_final_surv_estimates)
        kde_surv_variance[i] = np.var(kde_surv_estimates)
        km_surv_variance[i] = np.var(km_surv_estimates)
        
        # Compute MSE (mean squared error) - Survival
        hal_init_surv_mse[i] = np.mean((hal_init_surv_estimates - true_surv_value) ** 2)
        hal_final_surv_mse[i] = np.mean((hal_final_surv_estimates - true_surv_value) ** 2)
        kde_surv_mse[i] = np.mean((kde_surv_estimates - true_surv_value) ** 2)
        km_surv_mse[i] = np.mean((km_surv_estimates - true_surv_value) ** 2)
        
        # Extract density estimates at this evaluation point
        hal_init_dens_estimates = hal_init_dens_data[i, :]
        hal_final_dens_estimates = hal_final_dens_data[i, :]
        kde_dens_estimates = kde_dens_data[i, :]
        
        # True density value at this point
        true_dens_value = true_density[i]
        
        # Compute bias (mean estimate - true value) - Density
        hal_init_dens_bias[i] = np.mean(hal_init_dens_estimates) - true_dens_value
        hal_final_dens_bias[i] = np.mean(hal_final_dens_estimates) - true_dens_value
        kde_dens_bias[i] = np.mean(kde_dens_estimates) - true_dens_value
        
        # Compute variance - Density
        hal_init_dens_variance[i] = np.var(hal_init_dens_estimates)
        hal_final_dens_variance[i] = np.var(hal_final_dens_estimates)
        kde_dens_variance[i] = np.var(kde_dens_estimates)
        
        # Compute MSE (mean squared error) - Density
        hal_init_dens_mse[i] = np.mean((hal_init_dens_estimates - true_dens_value) ** 2)
        hal_final_dens_mse[i] = np.mean((hal_final_dens_estimates - true_dens_value) ** 2)
        kde_dens_mse[i] = np.mean((kde_dens_estimates - true_dens_value) ** 2)
    
    # Return all metrics
    metrics = {
        'eval_points': eval_points,
        # Survival metrics
        'hal_init_surv_bias': hal_init_surv_bias,
        'hal_final_surv_bias': hal_final_surv_bias,
        'kde_surv_bias': kde_surv_bias,
        'km_surv_bias': km_surv_bias,
        'hal_init_surv_variance': hal_init_surv_variance,
        'hal_final_surv_variance': hal_final_surv_variance,
        'kde_surv_variance': kde_surv_variance,
        'km_surv_variance': km_surv_variance,
        'hal_init_surv_mse': hal_init_surv_mse,
        'hal_final_surv_mse': hal_final_surv_mse,
        'kde_surv_mse': kde_surv_mse,
        'km_surv_mse': km_surv_mse,
        # Density metrics
        'hal_init_dens_bias': hal_init_dens_bias,
        'hal_final_dens_bias': hal_final_dens_bias,
        'kde_dens_bias': kde_dens_bias,
        'hal_init_dens_variance': hal_init_dens_variance,
        'hal_final_dens_variance': hal_final_dens_variance,
        'kde_dens_variance': kde_dens_variance,
        'hal_init_dens_mse': hal_init_dens_mse,
        'hal_final_dens_mse': hal_final_dens_mse,
        'kde_dens_mse': kde_dens_mse
    }
    
    return metrics

# New function to plot side-by-side bias, variance, and MSE with multiple smoothness levels
# New function to plot side-by-side bias, variance, and MSE with multiple smoothness levels
def plot_metrics_by_evaluation_point_multi_smooth(metrics_smooth0, metrics_smooth1, output_dir, n_samples):
    """
    Create side-by-side plots of bias, variance, and MSE across evaluation points
    for both smoothness levels but without final HAL density results.
    Keep final HAL results for survival function plots.
    
    Changes:
    - Removed overall title (suptitle)
    - Removed legend from each subplot
    - Added unified legend at the bottom of the plot
    - Ordered legend items as specified
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Get evaluation points
    eval_points = metrics_smooth0['eval_points']
    
    # ----- SURVIVAL FUNCTION PLOTS -----
    # Create figure with 3 subplots side by side for survival function
    fig_surv = plt.figure(figsize=(18, 6))
    gs_surv = GridSpec(1, 3, figure=fig_surv, wspace=0.3)
    
    # Plot 1: Bias across evaluation points for survival
    ax1_surv = fig_surv.add_subplot(gs_surv[0, 0])
    hal_init_s0_line, = ax1_surv.plot(eval_points, metrics_smooth0['hal_init_surv_bias'], 'b-', linewidth=2, markersize=6)
    hal_final_s0_line, = ax1_surv.plot(eval_points, metrics_smooth0['hal_final_surv_bias'], 'b--', linewidth=2, markersize=6)
    hal_init_s1_line, = ax1_surv.plot(eval_points, metrics_smooth1['hal_init_surv_bias'], 'g-', linewidth=2, markersize=6)
    hal_final_s1_line, = ax1_surv.plot(eval_points, metrics_smooth1['hal_final_surv_bias'], 'g--', linewidth=2, markersize=6)
    kde_line, = ax1_surv.plot(eval_points, metrics_smooth0['kde_surv_bias'], 'co-', linewidth=2, markersize=6)
    km_line, = ax1_surv.plot(eval_points, metrics_smooth0['km_surv_bias'], 'mo-', linewidth=2, markersize=6)
    
    ax1_surv.set_xlabel('Evaluation Point')
    ax1_surv.set_ylabel('Bias')
    ax1_surv.set_title('Bias Across Evaluation Points')
    ax1_surv.grid(True)
    
    # Plot 2: Variance across evaluation points for survival
    ax2_surv = fig_surv.add_subplot(gs_surv[0, 1])
    ax2_surv.plot(eval_points, metrics_smooth0['hal_init_surv_variance'], 'b-', linewidth=2, markersize=6)
    ax2_surv.plot(eval_points, metrics_smooth0['hal_final_surv_variance'], 'b--', linewidth=2, markersize=6)
    ax2_surv.plot(eval_points, metrics_smooth1['hal_init_surv_variance'], 'g-', linewidth=2, markersize=6)
    ax2_surv.plot(eval_points, metrics_smooth1['hal_final_surv_variance'], 'g--', linewidth=2, markersize=6)
    ax2_surv.plot(eval_points, metrics_smooth0['kde_surv_variance'], 'co-', linewidth=2, markersize=6)
    ax2_surv.plot(eval_points, metrics_smooth0['km_surv_variance'], 'mo-', linewidth=2, markersize=6)
    
    ax2_surv.set_xlabel('Evaluation Point')
    ax2_surv.set_ylabel('Variance')
    ax2_surv.set_title('Variance Across Evaluation Points')
    ax2_surv.grid(True)
    
    # Plot 3: MSE across evaluation points for survival
    ax3_surv = fig_surv.add_subplot(gs_surv[0, 2])
    ax3_surv.plot(eval_points, metrics_smooth0['hal_init_surv_mse'], 'b-', linewidth=2, markersize=6)
    ax3_surv.plot(eval_points, metrics_smooth0['hal_final_surv_mse'], 'b--', linewidth=2, markersize=6)
    ax3_surv.plot(eval_points, metrics_smooth1['hal_init_surv_mse'], 'g-', linewidth=2, markersize=6)
    ax3_surv.plot(eval_points, metrics_smooth1['hal_final_surv_mse'], 'g--', linewidth=2, markersize=6)
    ax3_surv.plot(eval_points, metrics_smooth0['kde_surv_mse'], 'co-', linewidth=2, markersize=6)
    ax3_surv.plot(eval_points, metrics_smooth0['km_surv_mse'], 'mo-', linewidth=2, markersize=6)
    
    ax3_surv.set_xlabel('Evaluation Point')
    ax3_surv.set_ylabel('MSE')
    ax3_surv.set_title('MSE Across Evaluation Points')
    ax3_surv.grid(True)
    
    # Add a unified legend at the bottom of the plot with the specified order
    fig_surv.legend(
        [hal_init_s0_line, hal_final_s0_line, hal_init_s1_line, hal_final_s1_line, kde_line, km_line],
        ['HAL Initial (smooth=0)', 'HAL Final (smooth=0)', 'HAL Initial (smooth=1)', 'HAL Final (smooth=1)', 'KDE', 'KM'],
        loc='lower center', bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=False
    )
    
    plt.tight_layout()
    
    # Save the figure for survival
    fig_surv.savefig(f"{output_dir}/survival_metrics_n{n_samples}.png", dpi=300, bbox_inches='tight')
    plt.close(fig_surv)
    
    # ----- DENSITY FUNCTION PLOTS -----
    # Create figure with 3 subplots side by side for density function - WITHOUT final HAL results
    fig_dens = plt.figure(figsize=(18, 6))
    gs_dens = GridSpec(1, 3, figure=fig_dens, wspace=0.3)
    
    # Plot 1: Bias across evaluation points for density - ONLY initial HAL results
    ax1_dens = fig_dens.add_subplot(gs_dens[0, 0])
    hal_init_s0_line_d, = ax1_dens.plot(eval_points, metrics_smooth0['hal_init_dens_bias'], 'b-', linewidth=2, markersize=6)
    hal_init_s1_line_d, = ax1_dens.plot(eval_points, metrics_smooth1['hal_init_dens_bias'], 'g-', linewidth=2, markersize=6)
    kde_line_d, = ax1_dens.plot(eval_points, metrics_smooth0['kde_dens_bias'], 'co-', linewidth=2, markersize=6)
    
    ax1_dens.set_xlabel('Evaluation Point')
    ax1_dens.set_ylabel('Bias')
    ax1_dens.set_title('Bias Across Evaluation Points')
    ax1_dens.grid(True)
    
    # Plot 2: Variance across evaluation points for density - ONLY initial HAL results
    ax2_dens = fig_dens.add_subplot(gs_dens[0, 1])
    ax2_dens.plot(eval_points, metrics_smooth0['hal_init_dens_variance'], 'b-', linewidth=2, markersize=6)
    ax2_dens.plot(eval_points, metrics_smooth1['hal_init_dens_variance'], 'g-', linewidth=2, markersize=6)
    ax2_dens.plot(eval_points, metrics_smooth0['kde_dens_variance'], 'co-', linewidth=2, markersize=6)
    
    ax2_dens.set_xlabel('Evaluation Point')
    ax2_dens.set_ylabel('Variance')
    ax2_dens.set_title('Variance Across Evaluation Points')
    ax2_dens.grid(True)
    
    # Plot 3: MSE across evaluation points for density - ONLY initial HAL results
    ax3_dens = fig_dens.add_subplot(gs_dens[0, 2])
    ax3_dens.plot(eval_points, metrics_smooth0['hal_init_dens_mse'], 'b-', linewidth=2, markersize=6)
    ax3_dens.plot(eval_points, metrics_smooth1['hal_init_dens_mse'], 'g-', linewidth=2, markersize=6)
    ax3_dens.plot(eval_points, metrics_smooth0['kde_dens_mse'], 'co-', linewidth=2, markersize=6)
    
    ax3_dens.set_xlabel('Evaluation Point')
    ax3_dens.set_ylabel('MSE')
    ax3_dens.set_title('MSE Across Evaluation Points')
    ax3_dens.grid(True)
    
    # Add a unified legend for density plots with the specified order
    fig_dens.legend(
        [hal_init_s0_line_d, hal_init_s1_line_d, kde_line_d],
        ['HAL Initial (smooth=0)', 'HAL Initial (smooth=1)', 'KDE'],
        loc='lower center', bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=False
    )
    
    plt.tight_layout()
    
    # Save the figure for density
    fig_dens.savefig(f"{output_dir}/density_metrics_n{n_samples}.png", dpi=300, bbox_inches='tight')
    plt.close(fig_dens)
    
    # Return the figure objects
    return fig_surv, fig_dens

# Function to plot summarized metrics across all sample sizes with multiple smoothness levels
def plot_sample_size_summary_multi_smooth(summary_data, output_dir):
    """
    Create plots of mean metrics across sample sizes for both density and survival functions
    with multiple smoothness levels but without final HAL density results.
    Creates both log-scale and normal-scale plots.
    
    Changes:
    - Removed overall title (suptitle)
    - Removed legend from each subplot
    - Added unified legend at the bottom of the plot
    - Ordered legend items as specified
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Convert to DataFrame for easier plotting
    summary_df = pd.DataFrame(summary_data)
    
    # ------ LOG SCALE PLOTS ------
    
    # ----- SURVIVAL FUNCTION SUMMARY PLOTS (LOG SCALE) -----
    fig_surv_log = plt.figure(figsize=(18, 6))
    gs_surv_log = GridSpec(1, 3, figure=fig_surv_log, wspace=0.3)
    
    # Plot 1: Mean Absolute Bias vs Sample Size (Survival)
    ax1_surv_log = fig_surv_log.add_subplot(gs_surv_log[0, 0])
    hal_init_s0_line, = ax1_surv_log.plot(summary_df['sample_size'], summary_df['hal_init_surv_mean_abs_bias_smooth0'], 'b-', 
                  linewidth=2, markersize=8)
    
    # Include final HAL results for survival
    hal_final_s0_line = None
    if 'hal_final_surv_mean_abs_bias_smooth0' in summary_df.columns:
        hal_final_s0_line, = ax1_surv_log.plot(summary_df['sample_size'], summary_df['hal_final_surv_mean_abs_bias_smooth0'], 'b--', 
                      linewidth=2, markersize=8)
    
    hal_init_s1_line, = ax1_surv_log.plot(summary_df['sample_size'], summary_df['hal_init_surv_mean_abs_bias_smooth1'], 'g-', 
                  linewidth=2, markersize=8)
    
    hal_final_s1_line = None
    if 'hal_final_surv_mean_abs_bias_smooth1' in summary_df.columns:
        hal_final_s1_line, = ax1_surv_log.plot(summary_df['sample_size'], summary_df['hal_final_surv_mean_abs_bias_smooth1'], 'g--', 
                      linewidth=2, markersize=8)
    
    kde_line, = ax1_surv_log.plot(summary_df['sample_size'], summary_df['kde_surv_mean_abs_bias'], 'co-', 
                  linewidth=2, markersize=8)
    
    km_line, = ax1_surv_log.plot(summary_df['sample_size'], summary_df['km_surv_mean_abs_bias'], 'mo-', 
                  linewidth=2, markersize=8)
    
    ax1_surv_log.set_xlabel('Sample Size')
    ax1_surv_log.set_ylabel('Mean Absolute Bias')
    ax1_surv_log.set_title('Mean Absolute Bias vs Sample Size')
    ax1_surv_log.set_xscale('log')
    ax1_surv_log.set_yscale('log')
    ax1_surv_log.grid(True, which="both", ls="-")
    
    # Plot 2: Mean Variance vs Sample Size (Survival)
    ax2_surv_log = fig_surv_log.add_subplot(gs_surv_log[0, 1])
    ax2_surv_log.plot(summary_df['sample_size'], summary_df['hal_init_surv_mean_variance_smooth0'], 'b-', 
                  linewidth=2, markersize=8)
    
    # Include final HAL results for survival
    if 'hal_final_surv_mean_variance_smooth0' in summary_df.columns:
        ax2_surv_log.plot(summary_df['sample_size'], summary_df['hal_final_surv_mean_variance_smooth0'], 'b--', 
                      linewidth=2, markersize=8)
    
    ax2_surv_log.plot(summary_df['sample_size'], summary_df['hal_init_surv_mean_variance_smooth1'], 'g-', 
                  linewidth=2, markersize=8)
    
    if 'hal_final_surv_mean_variance_smooth1' in summary_df.columns:
        ax2_surv_log.plot(summary_df['sample_size'], summary_df['hal_final_surv_mean_variance_smooth1'], 'g--', 
                      linewidth=2, markersize=8)
    
    ax2_surv_log.plot(summary_df['sample_size'], summary_df['kde_surv_mean_variance'], 'co-', 
                  linewidth=2, markersize=8)
    
    ax2_surv_log.plot(summary_df['sample_size'], summary_df['km_surv_mean_variance'], 'mo-', 
                  linewidth=2, markersize=8)
    
    ax2_surv_log.set_xlabel('Sample Size')
    ax2_surv_log.set_ylabel('Mean Variance')
    ax2_surv_log.set_title('Mean Variance vs Sample Size')
    ax2_surv_log.set_xscale('log')
    ax2_surv_log.set_yscale('log')
    ax2_surv_log.grid(True, which="both", ls="-")
    
    # Plot 3: Mean MSE vs Sample Size (Survival)
    ax3_surv_log = fig_surv_log.add_subplot(gs_surv_log[0, 2])
    ax3_surv_log.plot(summary_df['sample_size'], summary_df['hal_init_surv_mean_mse_smooth0'], 'b-', 
                  linewidth=2, markersize=8)
    
    # Include final HAL results for survival
    if 'hal_final_surv_mean_mse_smooth0' in summary_df.columns:
        ax3_surv_log.plot(summary_df['sample_size'], summary_df['hal_final_surv_mean_mse_smooth0'], 'b--', 
                      linewidth=2, markersize=8)
    
    ax3_surv_log.plot(summary_df['sample_size'], summary_df['hal_init_surv_mean_mse_smooth1'], 'g-', 
                  linewidth=2, markersize=8)
    
    if 'hal_final_surv_mean_mse_smooth1' in summary_df.columns:
        ax3_surv_log.plot(summary_df['sample_size'], summary_df['hal_final_surv_mean_mse_smooth1'], 'g--', 
                      linewidth=2, markersize=8)
    
    ax3_surv_log.plot(summary_df['sample_size'], summary_df['kde_surv_mean_mse'], 'co-', 
                  linewidth=2, markersize=8)
    
    ax3_surv_log.plot(summary_df['sample_size'], summary_df['km_surv_mean_mse'], 'mo-', 
                  linewidth=2, markersize=8)
    
    ax3_surv_log.set_xlabel('Sample Size')
    ax3_surv_log.set_ylabel('Mean MSE')
    ax3_surv_log.set_title('Mean MSE vs Sample Size')
    ax3_surv_log.set_xscale('log')
    ax3_surv_log.set_yscale('log')
    ax3_surv_log.grid(True, which="both", ls="-")
    
    # Create legend handles for the unified legend in the specified order
    legend_handles = [hal_init_s0_line]
    legend_labels = ['HAL Initial (smooth=0)']
    
    # Add final HAL lines to legend if they exist (maintaining the requested order)
    if hal_final_s0_line is not None:
        legend_handles.append(hal_final_s0_line)
        legend_labels.append('HAL Final (smooth=0)')
        
    # Add smoothness 1 lines
    legend_handles.append(hal_init_s1_line)
    legend_labels.append('HAL Initial (smooth=1)')
    
    if hal_final_s1_line is not None:
        legend_handles.append(hal_final_s1_line)
        legend_labels.append('HAL Final (smooth=1)')
    
    # Add KDE and KM lines in the specified order
    legend_handles.append(kde_line)
    legend_labels.append('KDE')
    
    legend_handles.append(km_line)
    legend_labels.append('Kaplan-Meier')
    
    # Add unified legend
    fig_surv_log.legend(
        legend_handles, legend_labels,
        loc='lower center', bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=False
    )
    
    plt.tight_layout()
    
    # Save the figure for survival
    fig_surv_log.savefig(f"{output_dir}/survival_metrics_by_sample_size_log.png", dpi=300, bbox_inches='tight')
    plt.close(fig_surv_log)
    
    # ----- DENSITY FUNCTION SUMMARY PLOTS (LOG SCALE) -----
    fig_dens_log = plt.figure(figsize=(18, 6))
    gs_dens_log = GridSpec(1, 3, figure=fig_dens_log, wspace=0.3)
    
    # Plot 1: Mean Absolute Bias vs Sample Size (Density)
    ax1_dens_log = fig_dens_log.add_subplot(gs_dens_log[0, 0])
    hal_init_s0_line_d, = ax1_dens_log.plot(summary_df['sample_size'], summary_df['hal_init_dens_mean_abs_bias_smooth0'], 'b-', 
                  linewidth=2, markersize=8)
    hal_init_s1_line_d, = ax1_dens_log.plot(summary_df['sample_size'], summary_df['hal_init_dens_mean_abs_bias_smooth1'], 'g-', 
                  linewidth=2, markersize=8)
    kde_line_d, = ax1_dens_log.plot(summary_df['sample_size'], summary_df['kde_dens_mean_abs_bias'], 'co-', 
                  linewidth=2, markersize=8)
    
    ax1_dens_log.set_xlabel('Sample Size')
    ax1_dens_log.set_ylabel('Mean Absolute Bias')
    ax1_dens_log.set_title('Mean Absolute Bias vs Sample Size')
    ax1_dens_log.set_xscale('log')
    ax1_dens_log.set_yscale('log')
    ax1_dens_log.grid(True, which="both", ls="-")
    
    # Plot 2: Mean Variance vs Sample Size (Density)
    ax2_dens_log = fig_dens_log.add_subplot(gs_dens_log[0, 1])
    ax2_dens_log.plot(summary_df['sample_size'], summary_df['hal_init_dens_mean_variance_smooth0'], 'b-', 
                  linewidth=2, markersize=8)
    ax2_dens_log.plot(summary_df['sample_size'], summary_df['hal_init_dens_mean_variance_smooth1'], 'g-', 
                  linewidth=2, markersize=8)
    ax2_dens_log.plot(summary_df['sample_size'], summary_df['kde_dens_mean_variance'], 'co-', 
                  linewidth=2, markersize=8)
    
    ax2_dens_log.set_xlabel('Sample Size')
    ax2_dens_log.set_ylabel('Mean Variance')
    ax2_dens_log.set_title('Mean Variance vs Sample Size')
    ax2_dens_log.set_xscale('log')
    ax2_dens_log.set_yscale('log')
    ax2_dens_log.grid(True, which="both", ls="-")
    
    # Plot 3: Mean MSE vs Sample Size (Density)
    ax3_dens_log = fig_dens_log.add_subplot(gs_dens_log[0, 2])
    ax3_dens_log.plot(summary_df['sample_size'], summary_df['hal_init_dens_mean_mse_smooth0'], 'b-', 
                  linewidth=2, markersize=8)
    ax3_dens_log.plot(summary_df['sample_size'], summary_df['hal_init_dens_mean_mse_smooth1'], 'g-', 
                  linewidth=2, markersize=8)
    ax3_dens_log.plot(summary_df['sample_size'], summary_df['kde_dens_mean_mse'], 'co-', 
                  linewidth=2, markersize=8)
    
    ax3_dens_log.set_xlabel('Sample Size')
    ax3_dens_log.set_ylabel('Mean MSE')
    ax3_dens_log.set_title('Mean MSE vs Sample Size')
    ax3_dens_log.set_xscale('log')
    ax3_dens_log.set_yscale('log')
    ax3_dens_log.grid(True, which="both", ls="-")
    
    # Add unified legend for density plots with the specified order
    fig_dens_log.legend(
        [hal_init_s0_line_d, hal_init_s1_line_d, kde_line_d],
        ['HAL Initial (smooth=0)', 'HAL Initial (smooth=1)', 'KDE'],
        loc='lower center', bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=False
    )
    
    plt.tight_layout()
    
    # Save the figure for density
    fig_dens_log.savefig(f"{output_dir}/density_metrics_by_sample_size_log.png", dpi=300, bbox_inches='tight')
    plt.close(fig_dens_log)
    
    # ------ NORMAL SCALE PLOTS ------
    
    # ----- SURVIVAL FUNCTION SUMMARY PLOTS (NORMAL SCALE) -----
    fig_surv_normal = plt.figure(figsize=(18, 6))
    gs_surv_normal = GridSpec(1, 3, figure=fig_surv_normal, wspace=0.3)
    
    # Plot 1: Mean Absolute Bias vs Sample Size (Survival) - NORMAL SCALE
    ax1_surv_normal = fig_surv_normal.add_subplot(gs_surv_normal[0, 0])
    hal_init_s0_line_n, = ax1_surv_normal.plot(summary_df['sample_size'], summary_df['hal_init_surv_mean_abs_bias_smooth0'], 'b-', 
                  linewidth=2, markersize=8)
    
    # Include final HAL results for survival
    hal_final_s0_line_n = None
    if 'hal_final_surv_mean_abs_bias_smooth0' in summary_df.columns:
        hal_final_s0_line_n, = ax1_surv_normal.plot(summary_df['sample_size'], summary_df['hal_final_surv_mean_abs_bias_smooth0'], 'b--', 
                      linewidth=2, markersize=8)
    
    hal_init_s1_line_n, = ax1_surv_normal.plot(summary_df['sample_size'], summary_df['hal_init_surv_mean_abs_bias_smooth1'], 'g-', 
                  linewidth=2, markersize=8)
    
    hal_final_s1_line_n = None
    if 'hal_final_surv_mean_abs_bias_smooth1' in summary_df.columns:
        hal_final_s1_line_n, = ax1_surv_normal.plot(summary_df['sample_size'], summary_df['hal_final_surv_mean_abs_bias_smooth1'], 'g--', 
                      linewidth=2, markersize=8)
    
    kde_line_n, = ax1_surv_normal.plot(summary_df['sample_size'], summary_df['kde_surv_mean_abs_bias'], 'co-', 
                  linewidth=2, markersize=8)
    
    km_line_n, = ax1_surv_normal.plot(summary_df['sample_size'], summary_df['km_surv_mean_abs_bias'], 'mo-', 
                  linewidth=2, markersize=8)
    
    ax1_surv_normal.set_xlabel('Sample Size')
    ax1_surv_normal.set_ylabel('Mean Absolute Bias')
    ax1_surv_normal.set_title('Mean Absolute Bias vs Sample Size')
    ax1_surv_normal.grid(True)
    
    # Plot 2: Mean Variance vs Sample Size (Survival) - NORMAL SCALE
    ax2_surv_normal = fig_surv_normal.add_subplot(gs_surv_normal[0, 1])
    ax2_surv_normal.plot(summary_df['sample_size'], summary_df['hal_init_surv_mean_variance_smooth0'], 'b-', 
                  linewidth=2, markersize=8)
    
    # Include final HAL results for survival
    if 'hal_final_surv_mean_variance_smooth0' in summary_df.columns:
        ax2_surv_normal.plot(summary_df['sample_size'], summary_df['hal_final_surv_mean_variance_smooth0'], 'b--', 
                      linewidth=2, markersize=8)
    
    ax2_surv_normal.plot(summary_df['sample_size'], summary_df['hal_init_surv_mean_variance_smooth1'], 'g-', 
                  linewidth=2, markersize=8)
    
    if 'hal_final_surv_mean_variance_smooth1' in summary_df.columns:
        ax2_surv_normal.plot(summary_df['sample_size'], summary_df['hal_final_surv_mean_variance_smooth1'], 'g--', 
                      linewidth=2, markersize=8)
    
    ax2_surv_normal.plot(summary_df['sample_size'], summary_df['kde_surv_mean_variance'], 'co-', 
                  linewidth=2, markersize=8)
    
    ax2_surv_normal.plot(summary_df['sample_size'], summary_df['km_surv_mean_variance'], 'mo-', 
                  linewidth=2, markersize=8)
    
    ax2_surv_normal.set_xlabel('Sample Size')
    ax2_surv_normal.set_ylabel('Mean Variance')
    ax2_surv_normal.set_title('Mean Variance vs Sample Size')
    ax2_surv_normal.grid(True)
    
    # Plot 3: Mean MSE vs Sample Size (Survival) - NORMAL SCALE
    ax3_surv_normal = fig_surv_normal.add_subplot(gs_surv_normal[0, 2])
    ax3_surv_normal.plot(summary_df['sample_size'], summary_df['hal_init_surv_mean_mse_smooth0'], 'b-', 
                  linewidth=2, markersize=8)
    
    # Include final HAL results for survival
    if 'hal_final_surv_mean_mse_smooth0' in summary_df.columns:
        ax3_surv_normal.plot(summary_df['sample_size'], summary_df['hal_final_surv_mean_mse_smooth0'], 'b--', 
                      linewidth=2, markersize=8)
    
    ax3_surv_normal.plot(summary_df['sample_size'], summary_df['hal_init_surv_mean_mse_smooth1'], 'g-', 
                  linewidth=2, markersize=8)
    
    if 'hal_final_surv_mean_mse_smooth1' in summary_df.columns:
        ax3_surv_normal.plot(summary_df['sample_size'], summary_df['hal_final_surv_mean_mse_smooth1'], 'g--', 
                      linewidth=2, markersize=8)
    
    ax3_surv_normal.plot(summary_df['sample_size'], summary_df['kde_surv_mean_mse'], 'co-', 
                  linewidth=2, markersize=8)
    
    ax3_surv_normal.plot(summary_df['sample_size'], summary_df['km_surv_mean_mse'], 'mo-', 
                  linewidth=2, markersize=8)
    
    ax3_surv_normal.set_xlabel('Sample Size')
    ax3_surv_normal.set_ylabel('Mean MSE')
    ax3_surv_normal.set_title('Mean MSE vs Sample Size')
    ax3_surv_normal.grid(True)
    
    # Create legend handles for the unified legend (normal scale) in the specified order
    legend_handles_n = [hal_init_s0_line_n]
    legend_labels_n = ['HAL Initial (smooth=0)']
    
    # Add final HAL lines to legend if they exist (maintaining the requested order)
    if hal_final_s0_line_n is not None:
        legend_handles_n.append(hal_final_s0_line_n)
        legend_labels_n.append('HAL Final (smooth=0)')
        
    # Add smoothness 1 lines
    legend_handles_n.append(hal_init_s1_line_n)
    legend_labels_n.append('HAL Initial (smooth=1)')
    
    if hal_final_s1_line_n is not None:
        legend_handles_n.append(hal_final_s1_line_n)
        legend_labels_n.append('HAL Final (smooth=1)')
    
    # Add KDE and KM lines in the specified order
    legend_handles_n.append(kde_line_n)
    legend_labels_n.append('KDE')
    
    legend_handles_n.append(km_line_n)
    legend_labels_n.append('Kaplan-Meier')
    
    # Add unified legend
    fig_surv_normal.legend(
        legend_handles_n, legend_labels_n,
        loc='lower center', bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=False
    )
    
    plt.tight_layout()
    
    # Save the figure for survival (normal scale)
    fig_surv_normal.savefig(f"{output_dir}/survival_metrics_by_sample_size_normal.png", dpi=300, bbox_inches='tight')
    plt.close(fig_surv_normal)
    
    # ----- DENSITY FUNCTION SUMMARY PLOTS (NORMAL SCALE) -----
    fig_dens_normal = plt.figure(figsize=(18, 6))
    gs_dens_normal = GridSpec(1, 3, figure=fig_dens_normal, wspace=0.3)
    
    # Plot 1: Mean Absolute Bias vs Sample Size (Density) - NORMAL SCALE
    ax1_dens_normal = fig_dens_normal.add_subplot(gs_dens_normal[0, 0])
    hal_init_s0_line_dn, = ax1_dens_normal.plot(summary_df['sample_size'], summary_df['hal_init_dens_mean_abs_bias_smooth0'], 'b-', 
                  linewidth=2, markersize=8)
    hal_init_s1_line_dn, = ax1_dens_normal.plot(summary_df['sample_size'], summary_df['hal_init_dens_mean_abs_bias_smooth1'], 'g-', 
                  linewidth=2, markersize=8)
    kde_line_dn, = ax1_dens_normal.plot(summary_df['sample_size'], summary_df['kde_dens_mean_abs_bias'], 'co-', 
                  linewidth=2, markersize=8)
    
    ax1_dens_normal.set_xlabel('Sample Size')
    ax1_dens_normal.set_ylabel('Mean Absolute Bias')
    ax1_dens_normal.set_title('Mean Absolute Bias vs Sample Size')
    ax1_dens_normal.grid(True)
    
    # Plot 2: Mean Variance vs Sample Size (Density) - NORMAL SCALE
    ax2_dens_normal = fig_dens_normal.add_subplot(gs_dens_normal[0, 1])
    ax2_dens_normal.plot(summary_df['sample_size'], summary_df['hal_init_dens_mean_variance_smooth0'], 'b-', 
                  linewidth=2, markersize=8)
    ax2_dens_normal.plot(summary_df['sample_size'], summary_df['hal_init_dens_mean_variance_smooth1'], 'g-', 
                  linewidth=2, markersize=8)
    ax2_dens_normal.plot(summary_df['sample_size'], summary_df['kde_dens_mean_variance'], 'co-', 
                  linewidth=2, markersize=8)
    
    ax2_dens_normal.set_xlabel('Sample Size')
    ax2_dens_normal.set_ylabel('Mean Variance')
    ax2_dens_normal.set_title('Mean Variance vs Sample Size')
    ax2_dens_normal.grid(True)
    
    # Plot 3: Mean MSE vs Sample Size (Density) - NORMAL SCALE
    ax3_dens_normal = fig_dens_normal.add_subplot(gs_dens_normal[0, 2])
    ax3_dens_normal.plot(summary_df['sample_size'], summary_df['hal_init_dens_mean_mse_smooth0'], 'b-', 
                  linewidth=2, markersize=8)
    ax3_dens_normal.plot(summary_df['sample_size'], summary_df['hal_init_dens_mean_mse_smooth1'], 'g-', 
                  linewidth=2, markersize=8)
    ax3_dens_normal.plot(summary_df['sample_size'], summary_df['kde_dens_mean_mse'], 'co-', 
                  linewidth=2, markersize=8)
    
    ax3_dens_normal.set_xlabel('Sample Size')
    ax3_dens_normal.set_ylabel('Mean MSE')
    ax3_dens_normal.set_title('Mean MSE vs Sample Size')
    ax3_dens_normal.grid(True)
    
    # Add unified legend for density normal scale with the specified order
    fig_dens_normal.legend(
        [hal_init_s0_line_dn, hal_init_s1_line_dn, kde_line_dn],
        ['HAL Initial (smooth=0)', 'HAL Initial (smooth=1)', 'KDE'],
        loc='lower center', bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=False
    )
    
    plt.tight_layout()
    
    # Save the figure for density (normal scale)
    fig_dens_normal.savefig(f"{output_dir}/density_metrics_by_sample_size_normal.png", dpi=300, bbox_inches='tight')
    plt.close(fig_dens_normal)
    
    return fig_surv_log, fig_dens_log, fig_surv_normal, fig_dens_normal
# Main function to process all sample sizes with multiple smoothness levels and generate comparison plots
def main():
    """Run comparison analysis for full data for all sample sizes with multiple smoothness levels."""
    # Create output directory
    metrics_output_dir = "metrics_fd_multi_smooth"
    plots_output_dir = "plots_fd_multi_smooth" 
    os.makedirs(metrics_output_dir, exist_ok=True)
    os.makedirs(plots_output_dir, exist_ok=True)
    
    # Process each sample size
    for n_samples in sample_sizes:
        print(f"\nProcessing sample size n={n_samples}...")
        
        # Define output comparison directory for this sample size
        out_comparison_dir = f"{comparison_base_dir}/comparison_output_fd"
        os.makedirs(out_comparison_dir, exist_ok=True)
        
        # Check if comparison data already exists for this sample size
        kde_density_file = f"{out_comparison_dir}/comparison_n{n_samples}_kde_density.csv"
        
        if not os.path.exists(kde_density_file):
            # Generate comparison data
            generate_comparison_data(n_samples, n_generate, evaluation_points, out_comparison_dir)
        else:
            print(f"Comparison data for n={n_samples} already exists, skipping generation.")
        
        # Load HAL and comparison data for both smoothness levels
        metrics_by_smoothness = {}
        
        for smoothness_order in smoothness_levels:
            # Load data for this smoothness level
            hal_data, comparison_data = load_data_fixed(hal_dir, out_comparison_dir, n_samples, smoothness_order, ridge)
            
            # Compute metrics
            metrics = compute_performance_metrics(hal_data, comparison_data)
            
            if metrics:
                # Save metrics to CSV
                metrics_df = pd.DataFrame({key: metrics[key] for key in metrics if key != 'eval_points'})
                metrics_df.index = metrics['eval_points']
                metrics_df.to_csv(f"{metrics_output_dir}/metrics_n{n_samples}_smooth{smoothness_order}.csv")
                
                # Store metrics for this smoothness level
                metrics_by_smoothness[smoothness_order] = metrics
        
        # If we have metrics for all smoothness levels, plot comparison charts
        if all(smooth in metrics_by_smoothness for smooth in smoothness_levels):
            # Plot metrics with both smoothness levels
            plot_metrics_by_evaluation_point_multi_smooth(
                metrics_by_smoothness[0], 
                metrics_by_smoothness[1], 
                plots_output_dir, 
                n_samples
            )
            
            # Update summary metrics for cross-sample size comparison
            sample_size_summary['sample_size'].append(n_samples)
            
            # Process metrics for smoothness=0
            metrics_smooth0 = metrics_by_smoothness[0]
            
            # Mean absolute bias - Survival (smoothness=0)
            sample_size_summary['hal_init_surv_mean_abs_bias_smooth0'].append(np.mean(np.abs(metrics_smooth0['hal_init_surv_bias'])))
            sample_size_summary['hal_final_surv_mean_abs_bias_smooth0'].append(np.mean(np.abs(metrics_smooth0['hal_final_surv_bias'])))
            sample_size_summary['km_surv_mean_abs_bias'].append(np.mean(np.abs(metrics_smooth0['km_surv_bias'])))
            sample_size_summary['kde_surv_mean_abs_bias'].append(np.mean(np.abs(metrics_smooth0['kde_surv_bias'])))
            
            # Mean variance - Survival (smoothness=0)
            sample_size_summary['hal_init_surv_mean_variance_smooth0'].append(np.mean(metrics_smooth0['hal_init_surv_variance']))
            sample_size_summary['hal_final_surv_mean_variance_smooth0'].append(np.mean(metrics_smooth0['hal_final_surv_variance']))
            sample_size_summary['km_surv_mean_variance'].append(np.mean(metrics_smooth0['km_surv_variance']))
            sample_size_summary['kde_surv_mean_variance'].append(np.mean(metrics_smooth0['kde_surv_variance']))
            
            # Mean MSE - Survival (smoothness=0)
            sample_size_summary['hal_init_surv_mean_mse_smooth0'].append(np.mean(metrics_smooth0['hal_init_surv_mse']))
            sample_size_summary['hal_final_surv_mean_mse_smooth0'].append(np.mean(metrics_smooth0['hal_final_surv_mse']))
            sample_size_summary['km_surv_mean_mse'].append(np.mean(metrics_smooth0['km_surv_mse']))
            sample_size_summary['kde_surv_mean_mse'].append(np.mean(metrics_smooth0['kde_surv_mse']))
            
            # Mean absolute bias - Density (smoothness=0)
            sample_size_summary['hal_init_dens_mean_abs_bias_smooth0'].append(np.mean(np.abs(metrics_smooth0['hal_init_dens_bias'])))
            sample_size_summary['kde_dens_mean_abs_bias'].append(np.mean(np.abs(metrics_smooth0['kde_dens_bias'])))
            
            # Mean variance - Density (smoothness=0)
            sample_size_summary['hal_init_dens_mean_variance_smooth0'].append(np.mean(metrics_smooth0['hal_init_dens_variance']))
            sample_size_summary['kde_dens_mean_variance'].append(np.mean(metrics_smooth0['kde_dens_variance']))
            
            # Mean MSE - Density (smoothness=0)
            sample_size_summary['hal_init_dens_mean_mse_smooth0'].append(np.mean(metrics_smooth0['hal_init_dens_mse']))
            sample_size_summary['kde_dens_mean_mse'].append(np.mean(metrics_smooth0['kde_dens_mse']))
            
            # Process metrics for smoothness=1
            metrics_smooth1 = metrics_by_smoothness[1]
            
            # Mean absolute bias - Survival (smoothness=1)
            sample_size_summary['hal_init_surv_mean_abs_bias_smooth1'].append(np.mean(np.abs(metrics_smooth1['hal_init_surv_bias'])))
            sample_size_summary['hal_final_surv_mean_abs_bias_smooth1'].append(np.mean(np.abs(metrics_smooth1['hal_final_surv_bias'])))
            
            # Mean variance - Survival (smoothness=1)
            sample_size_summary['hal_init_surv_mean_variance_smooth1'].append(np.mean(metrics_smooth1['hal_init_surv_variance']))
            sample_size_summary['hal_final_surv_mean_variance_smooth1'].append(np.mean(metrics_smooth1['hal_final_surv_variance']))
            
            # Mean MSE - Survival (smoothness=1)
            sample_size_summary['hal_init_surv_mean_mse_smooth1'].append(np.mean(metrics_smooth1['hal_init_surv_mse']))
            sample_size_summary['hal_final_surv_mean_mse_smooth1'].append(np.mean(metrics_smooth1['hal_final_surv_mse']))
            
            # Mean absolute bias - Density (smoothness=1)
            sample_size_summary['hal_init_dens_mean_abs_bias_smooth1'].append(np.mean(np.abs(metrics_smooth1['hal_init_dens_bias'])))
            
            # Mean variance - Density (smoothness=1)
            sample_size_summary['hal_init_dens_mean_variance_smooth1'].append(np.mean(metrics_smooth1['hal_init_dens_variance']))
            
            # Mean MSE - Density (smoothness=1)
            sample_size_summary['hal_init_dens_mean_mse_smooth1'].append(np.mean(metrics_smooth1['hal_init_dens_mse']))
        else:
            print(f"Warning: Could not find metrics for all smoothness levels for n={n_samples}")
    
    # Create summary plots across all sample sizes
    if len(sample_size_summary['sample_size']) > 0:
        # Save summary data
        summary_df = pd.DataFrame(sample_size_summary)
        summary_df.to_csv(f"{metrics_output_dir}/sample_size_summary_multi_smooth.csv", index=False)
        
        # Create summary plots
        plot_sample_size_summary_multi_smooth(sample_size_summary, plots_output_dir)
        
        print("\nSummary processing complete!")
        print(f"Metrics saved to {metrics_output_dir}")
        print(f"Plots saved to {plots_output_dir}")
        
        # Calculate convergence rates
        print("\nCalculating convergence rates...")
        if len(sample_size_summary['sample_size']) >= 3:  # Need at least 3 points for meaningful regression
            log_n = np.log(summary_df['sample_size'])
            
            # ----- SURVIVAL FUNCTION CONVERGENCE RATES -----
            print("\n----- SURVIVAL FUNCTION CONVERGENCE RATES -----")
            
            # HAL Initial (smooth=0) bias (survival)
            log_hal_init_surv_bias_smooth0 = np.log(summary_df['hal_init_surv_mean_abs_bias_smooth0'])
            hal_init_surv_bias_slope_smooth0, _, _, _, _ = linregress(log_n, log_hal_init_surv_bias_smooth0)
            print(f"HAL Initial (smooth=0) Survival Bias Convergence Rate: approximately n^({hal_init_surv_bias_slope_smooth0:.2f})")
            
            # HAL Initial (smooth=1) bias (survival)
            log_hal_init_surv_bias_smooth1 = np.log(summary_df['hal_init_surv_mean_abs_bias_smooth1'])
            hal_init_surv_bias_slope_smooth1, _, _, _, _ = linregress(log_n, log_hal_init_surv_bias_smooth1)
            print(f"HAL Initial (smooth=1) Survival Bias Convergence Rate: approximately n^({hal_init_surv_bias_slope_smooth1:.2f})")
            
            # HAL Final (smooth=0) bias (survival)
            log_hal_final_surv_bias_smooth0 = np.log(summary_df['hal_final_surv_mean_abs_bias_smooth0'])
            hal_final_surv_bias_slope_smooth0, _, _, _, _ = linregress(log_n, log_hal_final_surv_bias_smooth0)
            print(f"HAL Final (smooth=0) Survival Bias Convergence Rate: approximately n^({hal_final_surv_bias_slope_smooth0:.2f})")
            
            # HAL Final (smooth=1) bias (survival)
            log_hal_final_surv_bias_smooth1 = np.log(summary_df['hal_final_surv_mean_abs_bias_smooth1'])
            hal_final_surv_bias_slope_smooth1, _, _, _, _ = linregress(log_n, log_hal_final_surv_bias_smooth1)
            print(f"HAL Final (smooth=1) Survival Bias Convergence Rate: approximately n^({hal_final_surv_bias_slope_smooth1:.2f})")
            
            # KM bias (survival)
            log_km_surv_bias = np.log(summary_df['km_surv_mean_abs_bias'])
            km_surv_bias_slope, _, _, _, _ = linregress(log_n, log_km_surv_bias)
            print(f"Kaplan-Meier Bias Convergence Rate: approximately n^({km_surv_bias_slope:.2f})")
            
            # KDE bias (survival)
            log_kde_surv_bias = np.log(summary_df['kde_surv_mean_abs_bias'])
            kde_surv_bias_slope, _, _, _, _ = linregress(log_n, log_kde_surv_bias)
            print(f"KDE Survival Bias Convergence Rate: approximately n^({kde_surv_bias_slope:.2f})")
            
            # ----- DENSITY FUNCTION CONVERGENCE RATES -----
            print("\n----- DENSITY FUNCTION CONVERGENCE RATES -----")
            
            # HAL Initial (smooth=0) bias (density)
            log_hal_init_dens_bias_smooth0 = np.log(summary_df['hal_init_dens_mean_abs_bias_smooth0'])
            hal_init_dens_bias_slope_smooth0, _, _, _, _ = linregress(log_n, log_hal_init_dens_bias_smooth0)
            print(f"HAL Initial (smooth=0) Density Bias Convergence Rate: approximately n^({hal_init_dens_bias_slope_smooth0:.2f})")
            
            # HAL Initial (smooth=1) bias (density)
            log_hal_init_dens_bias_smooth1 = np.log(summary_df['hal_init_dens_mean_abs_bias_smooth1'])
            hal_init_dens_bias_slope_smooth1, _, _, _, _ = linregress(log_n, log_hal_init_dens_bias_smooth1)
            print(f"HAL Initial (smooth=1) Density Bias Convergence Rate: approximately n^({hal_init_dens_bias_slope_smooth1:.2f})")
            
            # KDE bias (density)
            log_kde_dens_bias = np.log(summary_df['kde_dens_mean_abs_bias'])
            kde_dens_bias_slope, _, _, _, _ = linregress(log_n, log_kde_dens_bias)
            print(f"KDE Density Bias Convergence Rate: approximately n^({kde_dens_bias_slope:.2f})")
            
            # Comparison of convergence rates
            print("\n----- CONVERGENCE RATE COMPARISON -----")
            
            print("\nSurvival Function Bias Convergence Rates:")
            methods = ["HAL Initial (smooth=0)", "HAL Initial (smooth=1)", 
                      "HAL Final (smooth=0)", "HAL Final (smooth=1)", 
                      "Kaplan-Meier", "KDE"]
            rates = [hal_init_surv_bias_slope_smooth0, hal_init_surv_bias_slope_smooth1,
                    hal_final_surv_bias_slope_smooth0, hal_final_surv_bias_slope_smooth1,
                    km_surv_bias_slope, kde_surv_bias_slope]
            
            for method, rate in sorted(zip(methods, rates), key=lambda x: x[1]):
                print(f"{method}: n^({rate:.2f})")
            
            print("\nDensity Function Bias Convergence Rates:")
            methods = ["HAL Initial (smooth=0)", "HAL Initial (smooth=1)", "KDE"]
            rates = [hal_init_dens_bias_slope_smooth0, hal_init_dens_bias_slope_smooth1, kde_dens_bias_slope]
            
            for method, rate in sorted(zip(methods, rates), key=lambda x: x[1]):
                print(f"{method}: n^({rate:.2f})")
        else:
            print("Need at least 3 sample sizes to calculate meaningful convergence rates")

if __name__ == "__main__":
    start_time = time.time()
    main()
    elapsed_time = time.time() - start_time
    print(f"\nTotal execution time: {elapsed_time:.2f} seconds ({elapsed_time/60:.2f} minutes)")


Processing sample size n=200...
Comparison data for n=200 already exists, skipping generation.
Computing metrics using 500 common experiments.
Computing metrics using 500 common experiments.


/var/folders/65/x30mj4jd3wb_rz8hlyy732_m0000gn/T/ipykernel_28609/331559904.py:465: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/var/folders/65/x30mj4jd3wb_rz8hlyy732_m0000gn/T/ipykernel_28609/331559904.py:516: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()



Processing sample size n=400...
Comparison data for n=400 already exists, skipping generation.
Computing metrics using 500 common experiments.
Computing metrics using 500 common experiments.


/var/folders/65/x30mj4jd3wb_rz8hlyy732_m0000gn/T/ipykernel_28609/331559904.py:465: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/var/folders/65/x30mj4jd3wb_rz8hlyy732_m0000gn/T/ipykernel_28609/331559904.py:516: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()



Processing sample size n=800...
Comparison data for n=800 already exists, skipping generation.
Computing metrics using 500 common experiments.
Computing metrics using 500 common experiments.


...


Processing sample size n=1600...
Comparison data for n=1600 already exists, skipping generation.
Computing metrics using 500 common experiments.
Computing metrics using 500 common experiments.


/var/folders/65/x30mj4jd3wb_rz8hlyy732_m0000gn/T/ipykernel_28609/331559904.py:465: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/var/folders/65/x30mj4jd3wb_rz8hlyy732_m0000gn/T/ipykernel_28609/331559904.py:516: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()



Processing sample size n=3200...
Comparison data for n=3200 already exists, skipping generation.
Computing metrics using 500 common experiments.
Computing metrics using 500 common experiments.


/var/folders/65/x30mj4jd3wb_rz8hlyy732_m0000gn/T/ipykernel_28609/331559904.py:465: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/var/folders/65/x30mj4jd3wb_rz8hlyy732_m0000gn/T/ipykernel_28609/331559904.py:516: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/var/folders/65/x30mj4jd3wb_rz8hlyy732_m0000gn/T/ipykernel_28609/331559904.py:671: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/var/folders/65/x30mj4jd3wb_rz8hlyy732_m0000gn/T/ipykernel_28609/331559904.py:736: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/var/folders/65/x30mj4jd3wb_rz8hlyy732_m0000gn/T/ipykernel_28609/331559904.py:864: UserWarning: This figure includes Axes that are n


Summary processing complete!
Metrics saved to metrics_fd_multi_smooth
Plots saved to plots_fd_multi_smooth

Calculating convergence rates...

----- SURVIVAL FUNCTION CONVERGENCE RATES -----
HAL Initial (smooth=0) Survival Bias Convergence Rate: approximately n^(-0.08)
HAL Initial (smooth=1) Survival Bias Convergence Rate: approximately n^(-0.44)
HAL Final (smooth=0) Survival Bias Convergence Rate: approximately n^(-0.02)
HAL Final (smooth=1) Survival Bias Convergence Rate: approximately n^(0.03)
Kaplan-Meier Bias Convergence Rate: approximately n^(-0.53)
KDE Survival Bias Convergence Rate: approximately n^(-0.38)

----- DENSITY FUNCTION CONVERGENCE RATES -----
HAL Initial (smooth=0) Density Bias Convergence Rate: approximately n^(-0.17)
HAL Initial (smooth=1) Density Bias Convergence Rate: approximately n^(-0.42)
KDE Density Bias Convergence Rate: approximately n^(-0.39)

----- CONVERGENCE RATE COMPARISON -----

Survival Function Bias Convergence Rates:
Kaplan-Meier: n^(-0.53)
HAL Ini